# Image embeddings → kNN → clustering

Real data: 2,000 photos from **CIFAR-10** (10 classes: planes, cars, birds, cats, deer, dogs, frogs, horses, ships, trucks).

1. Turn each image into a 512-number vector with **CLIP**
2. Look at the embedding space in 2D with **UMAP**
3. **kNN**: find similar images, classify images, search images with text
4. **Clustering**: group images without using any labels, then check how well it did

Runs fine on CPU (the embedding step takes a minute or two). Works in Google Colab too.

In [2]:
!pip install -q sentence-transformers umap-learn torchvision scikit-learn matplotlib


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\dbhagria\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


## 1. Load a real dataset
We take 200 random images per class so everything runs fast.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from torchvision.datasets import CIFAR10

ds = CIFAR10(root="data", train=False, download=True)   # ~170 MB download, first run only
classes = ds.classes
labels_all = np.array(ds.targets)

rng = np.random.default_rng(0)
idx = np.concatenate([rng.choice(np.where(labels_all == c)[0], 200, replace=False) for c in range(10)])
images = [ds[i][0] for i in idx]   # list of PIL images
y = labels_all[idx]                # true labels (only used for checking, never for clustering)
print(len(images), "images")

fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for ax, i in zip(axes.flat, rng.choice(len(images), 16, replace=False)):
    ax.imshow(images[i]); ax.set_title(classes[y[i]], fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

100.0%


Extracting data\cifar-10-python.tar.gz to data
2000 images


<Figure size 1200x350 with 16 Axes>

## 2. Images → embeddings
CLIP resizes each image internally. `normalize_embeddings=True` makes every vector length 1, so cosine similarity = dot product.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("clip-ViT-B-32")
emb = model.encode(images, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
print(emb.shape)   # (2000, 512)

## 3. See the embedding space
UMAP squashes 512 dimensions down to 2 while trying to keep neighbors as neighbors. Colors are the true labels: if CLIP did a good job, same-colored dots bunch up.

In [ ]:
import umap

xy = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=0).fit_transform(emb)

def scatter(ax, color_ids, title, cmap="tab10", vmax=9.5):
    ax.scatter(xy[:, 0], xy[:, 1], c=color_ids, cmap=cmap, s=6, vmin=-0.5, vmax=vmax)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])

fig, ax = plt.subplots(figsize=(8, 7))
scatter(ax, y, "CLIP embeddings in 2D (UMAP), colored by true label")
handles = [plt.Line2D([], [], marker="o", ls="", color=plt.cm.tab10(c), label=classes[c]) for c in range(10)]
ax.legend(handles=handles, fontsize=8, loc="best"); plt.show()

## 4. kNN
### 4a. Find similar images
First column is the query image, the rest are its 5 nearest neighbors with cosine distance `d` (0 = identical).

In [ ]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=6, metric="cosine").fit(emb)
queries = rng.choice(len(images), 5, replace=False)
dist, nbrs = nn.kneighbors(emb[queries])   # nearest neighbor #0 is the query itself

fig, axes = plt.subplots(5, 6, figsize=(9, 8.5))
for r, (row, d) in enumerate(zip(nbrs, dist)):
    for c, (j, dj) in enumerate(zip(row, d)):
        ax = axes[r, c]; ax.imshow(images[j]); ax.axis("off")
        ax.set_title(f"QUERY\n{classes[y[j]]}" if c == 0 else f"{classes[y[j]]}\nd={dj:.2f}", fontsize=8)
plt.tight_layout(); plt.show()

### 4b. kNN as a classifier
Hide the labels of 30% of the images, then label each one by majority vote of its 10 nearest labeled neighbors. Random guessing would get 10%.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import ConfusionMatrixDisplay

Xtr, Xte, ytr, yte = train_test_split(emb, y, test_size=0.3, stratify=y, random_state=0)
knn = KNeighborsClassifier(n_neighbors=10, metric="cosine").fit(Xtr, ytr)
print(f"kNN accuracy: {knn.score(Xte, yte):.1%}")

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_estimator(knn, Xte, yte, display_labels=classes,
                                      xticks_rotation=45, cmap="Blues", colorbar=False, ax=ax)
ax.set_title("Where kNN gets confused"); plt.tight_layout(); plt.show()

### 4c. Bonus: search images with text
CLIP puts text and images in the *same* space, so a sentence's embedding can be used as a kNN query against the images.

In [ ]:
for text in ["a red sports car", "a cat lying down", "a ship on the water", "an animal standing in a field"]:
    q = model.encode([text], normalize_embeddings=True)
    top = np.argsort(-(emb @ q.T).ravel())[:6]   # highest cosine similarity first
    fig, axes = plt.subplots(1, 6, figsize=(9, 1.9)); fig.suptitle(f'"{text}"', fontsize=10)
    for ax, j in zip(axes, top): ax.imshow(images[j]); ax.axis("off")
    plt.show()

## 5. Clustering (no labels used)
### 5a. K-means with k = 10
We tell it to find 10 groups. Cluster IDs are arbitrary, so colors won't match between the two plots; what matters is whether the *shapes* of the groups match. ARI scores the match: 1.0 = perfect, 0 = random.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

clusters = KMeans(n_clusters=10, n_init=10, random_state=0).fit_predict(emb)
print("Adjusted Rand Index vs true labels:", round(adjusted_rand_score(y, clusters), 3))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
scatter(axes[0], y, "True labels")
scatter(axes[1], clusters, "K-means clusters (found without labels)")
plt.tight_layout(); plt.show()

### 5b. What's inside each cluster?
Each row is one cluster: 8 random members, plus which true class dominates it and how pure it is.

In [ ]:
fig, axes = plt.subplots(10, 8, figsize=(8, 12))
for k in range(10):
    members = np.where(clusters == k)[0]
    counts = np.bincount(y[members], minlength=10)
    axes[k, 0].set_title(f"cluster {k}: {counts.max() / len(members):.0%} {classes[counts.argmax()]}  (n={len(members)})",
                         loc="left", fontsize=8)
    for c, ax in enumerate(axes[k]):
        ax.axis("off")
        if c < len(members):
            ax.imshow(images[rng.choice(members)])
plt.tight_layout(); plt.show()

### 5c. HDBSCAN: let the algorithm decide how many clusters
Clustering in 512-D is mushy, so we first reduce to 10-D with UMAP. HDBSCAN marks points that don't fit anywhere as noise (gray).

In [ ]:
from sklearn.cluster import HDBSCAN

emb10 = umap.UMAP(n_components=10, n_neighbors=15, metric="cosine", random_state=0).fit_transform(emb)
hl = HDBSCAN(min_cluster_size=30).fit_predict(emb10)
print("clusters found:", hl.max() + 1, "| noise points:", (hl == -1).sum())

fig, ax = plt.subplots(figsize=(8, 7))
noise = hl == -1
ax.scatter(xy[noise, 0], xy[noise, 1], c="lightgray", s=5, label="noise")
ax.scatter(xy[~noise, 0], xy[~noise, 1], c=hl[~noise], cmap="tab20", s=6)
ax.set_title(f"HDBSCAN found {hl.max() + 1} clusters on its own"); ax.set_xticks([]); ax.set_yticks([])
ax.legend(); plt.show()

## Try next
- Swap CIFAR-10 for your own folder: `images = [Image.open(p).convert("RGB") for p in glob.glob("my_photos/*.jpg")]`
- Try a different model, e.g. `SentenceTransformer("clip-ViT-L-14")` (bigger, usually better)
- Change `n_clusters` or `min_cluster_size` and watch the groups split and merge